In [3]:
import gradio as gr
import json
import os

# 1. LOAD DATA
DATA_PATH = "data/passages.jsonl"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Please ensure data/passages.jsonl exists in your root folder.")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    passages = [json.loads(line) for line in f]

# 2. DIAGNOSTIC LOGIC
def diagnose(read_score, listen_score):
    read_score = float(read_score)
    listen_score = float(listen_score)

    if listen_score >= 0.7 and read_score < 0.7:
        return "🔤 **Decoding Gap:** You understand the story when you hear it, but struggle to read the words. Pagezy will help you practice listening while reading."
    elif listen_score < 0.7 and read_score < 0.7:
        return "🧠 **Comprehension Gap:** The story is hard to understand in both modes. Pagezy will help break down the vocabulary and context."
    else:
        return "✅ **On Track:** Great job! You are comprehending the text well."

# 3. UI FUNCTIONS
def load_passage(passage_id):
    p = next((x for x in passages if x["id"] == passage_id), passages[0])
    q = p["quiz"][0]
    return (
        p["read_text"],
        f"**Question:** {q['question']}",
        gr.Radio(choices=q["options"], label="Your Answer", value=None)
    )

def check_answer(passage_id, choice, read_score, listen_score):
    if not choice:
        return "⚠️ Please select an answer first.", ""

    p = next((x for x in passages if x["id"] == passage_id), passages[0])
    q = p["quiz"][0]
    diagnosis = diagnose(read_score, listen_score)

    if choice == q["correct"]:
        return f"🎉 **Correct!** {q['explanation']}", diagnosis
    else:
        # TODO: Replace this with your actual AI model call later
        mock_hint = f"💡 **Tutor Hint:** Look closely at the second paragraph. The text mentions that '{q['correct']}' is the key reason."
        return f"🤔 **Not quite.** {mock_hint}\n\n(Try again!)", diagnosis

# 4. BUILD THE GRADIO UI
with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📖 Pagezy MVP: Read. Assess. Learn.")
    gr.Markdown("A simple tool to find out *why* a student is struggling with reading.")

    with gr.Row():
        with gr.Column(scale=2):
            passage_dropdown = gr.Dropdown(
                choices=[p["id"] for p in passages],
                value=passages[0]["id"],
                label="1. Choose a Story"
            )
            read_text = gr.Textbox(label="2. Read the Story", lines=6, interactive=False)

            gr.Markdown("### 3. How well did you understand it?")
            with gr.Row():
                read_slider = gr.Slider(0, 1, step=0.1, value=0.5, label="By READING it")
                listen_slider = gr.Slider(0, 1, step=0.1, value=0.8, label="If SOMEONE READ it to you")

        with gr.Column(scale=1):
            question_box = gr.Markdown(label="4. Quick Quiz")
            answer_radio = gr.Radio(label="Your Answer", choices=[])
            submit_btn = gr.Button("Check Answer", variant="primary")

            feedback_box = gr.Markdown(label="Feedback / Hint")
            diagnosis_box = gr.Markdown(label="Pagezy Diagnosis")

    # 5. WIRE IT UP
    passage_dropdown.change(
        load_passage,
        inputs=[passage_dropdown],
        outputs=[read_text, question_box, answer_radio]
    )

    submit_btn.click(
        check_answer,
        inputs=[passage_dropdown, answer_radio, read_slider, listen_slider],
        outputs=[feedback_box, diagnosis_box]
    )

# 6. LAUNCH
if __name__ == "__main__":
    demo.launch(share=True)

/tmp/ipykernel_20313/819323625.py:51: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://6f02f6101749ee9231.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
